In [ ]:
!pip install -q ultralytics
!pip install -q torchvision torch
!pip install -q pycocotools
!pip install -q matplotlib opencv-python tqdm

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.7 MB/s eta 0:00:00


MessageError: Error: credential propagation was unsuccessful

# EDA

In [ ]:
import os
from pathlib import Path
import shutil
import cv2
import yaml
import torch
import torch.nn as nn
from tqdm import tqdm
from ultralytics import YOLO
from collections import defaultdict
import matplotlib.pyplot as plt
import random
import numpy as np


In [ ]:
# 1. SETUP PATHS (Sesuaikan BASE_DIR dengan lokasi folder VMERGED_YOLO kamu)
BASE_DIR = Path("/content/drive/MyDrive/VMERGED_YOLO_1")

# Path langsung ke folder labels sesuai struktur gambar
LBL_TRAIN = BASE_DIR / "labels" / "train"
LBL_VAL   = BASE_DIR / "labels" / "val"
LBL_TEST  = BASE_DIR / "labels" / "test"

IMG_TRAIN = BASE_DIR / "images" / "train"
IMG_VAL   = BASE_DIR / "images" / "val"
IMG_TEST  = BASE_DIR / "images" / "test"

print(f"Image Train Path: {IMG_TRAIN}")
print(f"Image Validation Path: {IMG_VAL}")
print(f"Image Test Path: {IMG_TEST}")

CLASSES = ["motorcycle", "car", "bus", "truck"]

In [ ]:
def load_yolo_labels(label_dir):
    class_counts = defaultdict(int)
    box_sizes = []
    aspect_ratios = []

    if not label_dir.exists():
        print(f"Warning: Folder {label_dir} tidak ditemukan.")
        return class_counts, box_sizes, aspect_ratios

    for lbl_file in label_dir.glob("*.txt"):
        with open(lbl_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue

                cls, xc, yc, w, h = map(float, parts)
                cls = int(cls)

                class_counts[cls] += 1
                box_sizes.append(w * h) # Area relatif (0-1)
                aspect_ratios.append(w / h if h > 0 else 0)

    return class_counts, box_sizes, aspect_ratios

In [ ]:
print("Analyzing dataset structure...")
train_cls, train_box, train_ar = load_yolo_labels(LBL_TRAIN)
val_cls, val_box, val_ar       = load_yolo_labels(LBL_VAL)
test_cls, test_box, test_ar    = load_yolo_labels(LBL_TEST)

In [ ]:
def print_summary(name, counts):
    total = sum(counts.values())
    print(f"\n--- {name} Summary ---")
    print(f"Total Instances: {total}")
    for cls_idx, count in sorted(counts.items()):
        cls_name = CLASSES[cls_idx] if cls_idx < len(CLASSES) else f"ID {cls_idx}"
        percentage = (count/total)*100 if total > 0 else 0
        print(f"- {cls_name}: {count} ({percentage:.2f}%)")

print_summary("TRAIN", train_cls)
print_summary("VALIDATION", val_cls)
print_summary("TEST", test_cls)

In [ ]:
import pandas as pd
import seaborn as sns

def create_class_df(counts_dict, dataset_name, classes):
    data = []
    for cls_idx, count in sorted(counts_dict.items()):
        cls_name = classes[cls_idx] if cls_idx < len(classes) else f"ID {cls_idx}"
        data.append({'Dataset': dataset_name, 'Class': cls_name, 'Count': count})
    return pd.DataFrame(data)

train_df = create_class_df(train_cls, 'Train', CLASSES)
val_df = create_class_df(val_cls, 'Validation', CLASSES)
test_df = create_class_df(test_cls, 'Test', CLASSES)

combined_df = pd.concat([train_df, val_df, test_df])

plt.figure(figsize=(12, 6))
sns.barplot(x='Class', y='Count', hue='Dataset', data=combined_df)
plt.title('Class Distribution Across Datasets')
plt.xlabel('Object Class')
plt.ylabel('Number of Instances')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
def plot_distribution(data, title, xlabel, bins=50):
    plt.figure(figsize=(10, 5))
    sns.histplot(data, bins=bins, kde=True)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel('Frequency')
    plt.grid(axis='y', alpha=0.75)
    plt.tight_layout()
    plt.show()

print("\n--- Bounding Box Size (Area) Distribution ---")
plot_distribution(train_box, 'Train Bounding Box Area Distribution (Normalized)', 'Normalized Area (0-1)')
plot_distribution(val_box, 'Validation Bounding Box Area Distribution (Normalized)', 'Normalized Area (0-1)')
plot_distribution(test_box, 'Test Bounding Box Area Distribution (Normalized)', 'Normalized Area (0-1)')

print("\n--- Bounding Box Aspect Ratio Distribution (Width/Height) ---")
plot_distribution(train_ar, 'Train Bounding Box Aspect Ratio Distribution', 'Aspect Ratio (Width/Height)', bins=100)
plot_distribution(val_ar, 'Validation Bounding Box Aspect Ratio Distribution', 'Aspect Ratio (Width/Height)', bins=100)
plot_distribution(test_ar, 'Test Bounding Box Aspect Ratio Distribution', 'Aspect Ratio (Width/Height)', bins=100)

In [ ]:
def visualize_samples(img_dir, lbl_dir, n=5):
    # Check if the image directory exists
    if not img_dir.exists():
        print(f"Warning: Image directory {img_dir} not found. Skipping visualization.")
        return

    # Get all image files and ensure there are enough samples
    all_imgs = list(img_dir.glob("*.jpg"))
    if not all_imgs:
        print(f"No .jpg images found in {img_dir}. Skipping visualization.")
        return
    if len(all_imgs) < n:
        print(f"Warning: Not enough images ({len(all_imgs)}) in {img_dir} to visualize {n} samples. Visualizing all available.")
        n = len(all_imgs)

    imgs = random.sample(all_imgs, n)

    plt.figure(figsize=(15,3*n))
    for i, img_path in enumerate(imgs):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        lbl_path = lbl_dir / img_path.name.replace(".jpg", ".txt")
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.split()
                    if len(parts) < 5: continue
                    cls, xc, yc, bw, bh = map(float, parts)

                    x1 = int((xc - bw/2) * w)
                    y1 = int((yc - bh/2) * h)
                    x2 = int((xc + bw/2) * w)
                    y2 = int((yc + bh/2) * h)
                    cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
                    cls_name = CLASSES[int(cls)] if int(cls) < len(CLASSES) else f"ID {int(cls)}"
                    cv2.putText(img, cls_name, (x1, y1-5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)

        plt.subplot(n,1,i+1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(img_path.name)

    plt.tight_layout()
    plt.show()

visualize_samples(IMG_TRAIN, LBL_TRAIN, n=5)

In [ ]:
#Resplit sets
OUT_ROOT = Path("/content/drive/MyDrive/VMERGED_RES")

VAL_RATIO = 0.10
SEED = 42
random.seed(SEED)

NAMES = ["motorcycle", "car", "bus", "truck"]
IMG_EXTS = {".jpg", ".jpeg", ".png"}

def ensure_dir(p):
    p.mkdir(parents=True, exist_ok=True)

def list_images(folder):
    imgs = []
    for p in folder.rglob("*"):
        if p.suffix.lower() in IMG_EXTS:
            imgs.append(p)
    return imgs

def read_label(path):
    out = []
    if not path.exists():
        return out
    for line in path.read_text().splitlines():
        parts = line.split()
        if len(parts) >= 5:
            out.append(int(parts[0]))
    return out

# Load pool (train+val lama)
train_imgs = list_images(BASE_DIR/"images/train")
val_imgs   = list_images(BASE_DIR/"images/val")
test_imgs  = list_images(BASE_DIR/"images/test")

pool_imgs = train_imgs + val_imgs

samples = []
for img in pool_imgs:
    lbl = BASE_DIR/"labels"/("train" if img in train_imgs else "val")/(img.stem+".txt")
    classes = read_label(lbl)
    if len(classes) > 0:
        samples.append((img, lbl, classes))

# Hitung total per class
counts = defaultdict(int)
for _,_,cls in samples:
    for c in cls:
        counts[c]+=1

targets = {c:int(counts[c]*VAL_RATIO) for c in counts}
print("Target val bbox per class:", targets)

val_set = []
train_set = []
val_counts = defaultdict(int)

for img,lbl,cls in samples:
    put_in_val = False
    for c in cls:
        if val_counts[c] < targets[c]:
            put_in_val = True
    if put_in_val:
        val_set.append((img,lbl))
        for c in cls:
            val_counts[c]+=1
    else:
        train_set.append((img,lbl))

print("Final val bbox counts:", dict(val_counts))

# Copy files
for split in ["train","val","test"]:
    ensure_dir(OUT_ROOT/"images"/split)
    ensure_dir(OUT_ROOT/"labels"/split)

def copy_samples(samples, split):
    for img,lbl in tqdm(samples):
        shutil.copy2(img, OUT_ROOT/"images"/split/img.name)
        shutil.copy2(lbl, OUT_ROOT/"labels"/split/(img.stem+".txt"))

copy_samples(train_set,"train")
copy_samples(val_set,"val")

# copy test apa adanya
for img in test_imgs:
    lbl = BASE_DIR/"labels/test"/(img.stem+".txt")
    shutil.copy2(img, OUT_ROOT/"images/test"/img.name)
    if lbl.exists():
        shutil.copy2(lbl, OUT_ROOT/"labels/test"/(img.stem+".txt"))

print("DONE. Saved to:", OUT_ROOT)


# Train ulang dari sini

In [ ]:
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')
import os
from pathlib import Path
import shutil
import cv2
import yaml
import torch
import torch.nn as nn
from tqdm import tqdm
from ultralytics import YOLO
from collections import defaultdict
import matplotlib.pyplot as plt
import random
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.1 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
#Check current dataset distribution
CURR_DIR = Path("/content/drive/MyDrive/VMERGED_RES")
'''
LBL_TRAIN = CURR_DIR / "labels" / "train"
LBL_VAL   = CURR_DIR / "labels" / "val"
LBL_TEST  = CURR_DIR / "labels" / "test"

train_cls, train_box, train_ar = load_yolo_labels(LBL_TRAIN)
val_cls, val_box, val_ar       = load_yolo_labels(LBL_VAL)
test_cls, test_box, test_ar    = load_yolo_labels(LBL_TEST)

print_summary("TRAIN", train_cls)
print_summary("VALIDATION", val_cls)
print_summary("TEST", test_cls)
'''

'\nLBL_TRAIN = CURR_DIR / "labels" / "train"\nLBL_VAL   = CURR_DIR / "labels" / "val"\nLBL_TEST  = CURR_DIR / "labels" / "test"\n\ntrain_cls, train_box, train_ar = load_yolo_labels(LBL_TRAIN)\nval_cls, val_box, val_ar       = load_yolo_labels(LBL_VAL)\ntest_cls, test_box, test_ar    = load_yolo_labels(LBL_TEST)\n\nprint_summary("TRAIN", train_cls)\nprint_summary("VALIDATION", val_cls)\nprint_summary("TEST", test_cls)\n'

In [ ]:
'''
 (OUT_ROOT/"data.yaml").write_text(f"""
path: {OUT_ROOT}
train: images/train
val: images/val
test: images/test
names:
  0: motorcycle
  1: car
  2: bus
  3: truck
""")
'''

'\n (OUT_ROOT/"data.yaml").write_text(f"""\npath: {OUT_ROOT}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: motorcycle\n  1: car\n  2: bus\n  3: truck\n""")\n'

In [ ]:
# setup
BASE = Path("/content/drive/MyDrive")

DATA_YAML = CURR_DIR / "data.yaml"   # dataset resplit kamu
RUNS_DIR  = BASE / "runs_yolo2"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
#san check
from ultralytics import YOLO

model = YOLO("yolo11m.pt")  # atau yolo11s.pt kalau GPU kuat

model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=1280,
    conf=0.001,
    iou=0.6,
)

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11m summary (fused): 125 layers, 20,091,712 parameters, 0 gradients, 68.0 GFLOPs
val: Fast image access ✅ (ping: 0.8±0.6 ms, read: 0.7±0.5 MB/s, size: 337.0 KB)
val: Scanning /content/drive/MyDrive/VMERGED_RES/labels/val.cache... 941 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 941/941 263.1Mit/s 0.0s
val: /content/drive/MyDrive/VMERGED_RES/images/val/visdrone__9999981_00000_d_0000047.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 59/59 1.6it/s 36.8s
                   all        941      35790     0.0167      0.111    0.00968    0.00391
                person        689       3942     0.0586      0.178     0.0306     0.0107
               bicycle        932      28994    0.00202   0.000172    0.00289    0.00174
                   car        372        596    0.00521      0.263    0.00397  

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b0bf0340a70>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [ ]:
#Stage 1 - Baseline training (seadanya)
from ultralytics import YOLO

model = YOLO("yolo11m.pt")  # upgrade ke yolo11l.pt kalau sanggup

results1 = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=1280,          # 960/1024 oke; 960 sering lebih stabil + hemat VRAM
    batch=4,            # kalau OOM turunin ke 6/4
    optimizer="SGD",
    lr0=0.005,
    momentum=0.937,
    weight_decay=5e-4,
    warmup_epochs=3,

    # augment yang aman untuk object detection
    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=15,    # matiin mosaic di akhir biar bbox lebih “rapi”
    fliplr=0.5,

    patience=20,        # early stopping berdasar val
    device=0,
    project=str(RUNS_DIR),
    name="stage1_y11m_1280",
    exist_ok=True,
)


Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/VMERGED_RES/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=stage1_y11m_1280, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_m

In [ ]:
#Stage 2a - Focused -> Buat copyan untuk truck dan motou
import shutil
from tqdm import tqdm

SRC = BASE / "VMERGED_RES"
FOCUS = BASE / "VMERGED_TM_FOCUS_1"
'''
# class index kamu: 0 motorcycle, 1 car, 2 bus, 3 truck
FOCUS_CLASSES = {0, 3}

def ensure(p): p.mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    ensure(FOCUS / "images" / split)
    ensure(FOCUS / "labels" / split)

# fokus hanya dari TRAIN, VAL & TEST tetap sama (biar evaluasi konsisten)
train_label_dir = SRC / "labels/train"
train_image_dir = SRC / "images/train"

kept = 0
for lbl in tqdm(list(train_label_dir.glob("*.txt")), desc="Build focus train"):
    txt = lbl.read_text().strip().splitlines()
    cls_set = set()
    for line in txt:
        parts = line.split()
        if len(parts) >= 1:
            cls_set.add(int(float(parts[0])))
    if cls_set & FOCUS_CLASSES:
        img = train_image_dir / (lbl.stem + ".jpg")
        if not img.exists():
            # kalau ext beda, cari
            found = list(train_image_dir.glob(lbl.stem + ".*"))
            if not found:
                continue
            img = found[0]
        shutil.copy2(img,  FOCUS / "images/train" / img.name)
        shutil.copy2(lbl,  FOCUS / "labels/train" / lbl.name)
        kept += 1

# copy val/test apa adanya
for split in ["val", "test"]:
    for img in tqdm(list((SRC / f"images/{split}").glob("*.*")), desc=f"Copy {split}"):
        lbl = (SRC / f"labels/{split}" / (img.stem + ".txt"))
        shutil.copy2(img, FOCUS / f"images/{split}" / img.name)
        if lbl.exists():
            shutil.copy2(lbl, FOCUS / f"labels/{split}" / lbl.name)

# write yaml
(FOCUS / "data.yaml").write_text(f"""\
path: {FOCUS}
train: images/train
val: images/val
test: images/test
names:
  0: motorcycle
  1: car
  2: bus
  3: truck
""")

print("Focus train images kept:", kept)
print("Focus yaml:", FOCUS / "data.yaml")
'''

Copy test: 100%|██████████| 1591/1591 [07:30<00:00,  3.53it/s]

Focus train images kept: 4674
Focus yaml: /content/drive/MyDrive/VMERGED_TM_FOCUS_1/data.yaml


In [ ]:
#Stage 2b - Fine-tuning hasil stage 1
from ultralytics import YOLO

best1 = RUNS_DIR / "stage1_y11m_1280" / "weights" / "best.pt"
model2 = YOLO(str(best1))

results2 = model2.train(
    data=str(FOCUS / "data.yaml"),
    epochs=40,
    imgsz=1280,
    batch=4,
    optimizer="SGD",
    lr0=0.001,          # kecil biar nggak “lupa”
    warmup_epochs=2,
    close_mosaic=10,     # focus dataset biasanya lebih kecil → mosaic ga wajib
    mosaic=0.5,
    mixup=0.0,
    patience=10,
    device=0,
    project=str(RUNS_DIR),
    name="stage2_y11m_1280_focusMT_2",
    exist_ok=True,
)


Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/VMERGED_TM_FOCUS_1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/runs_yolo2/stage1_y11m_1280/weights/best.pt, momentum=0.937, mosaic=0.3, multi_scale=0.0, name=stage2_y11m_1280_focusMT, nb

In [ ]:
#Eval
from ultralytics import YOLO
import numpy as np

best2 = RUNS_DIR / "stage2_y11m_1280_focusMT" / "weights" / "best.pt"
m = YOLO(str(best2))

metrics = m.val(
    data=str(DATA_YAML),   # evaluasi balik ke dataset utama
    split="test",
    imgsz=1280,
    conf=0.001,
    iou=0.6,
    verbose=True
)



Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11m summary (fused): 126 layers, 20,033,116 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.0 ms, read: 95.4±45.0 MB/s, size: 160.5 KB)
val: Scanning /content/drive/MyDrive/VMERGED_RES/labels/test.cache... 1591 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1591/1591 444.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 100/100 3.3it/s 30.1s
                   all       1591      45513      0.728      0.629       0.67       0.44
            motorcycle        802       5883      0.616      0.515      0.514      0.231
                   car       1546      34023      0.892      0.844      0.899      0.613
                   bus        843       2945      0.789      0.583      0.683      0.518
                 truck        753       2662      0.615      0.575      0.583        0.4
Speed: 1.7

In [ ]:
#Print rapi
import numpy as np

print("\n===== OVERALL METRICS (2 Stage) =====")

map50 = np.mean(metrics.box.map50)
map5095 = np.mean(metrics.box.map)
precision = np.mean(metrics.box.mp)
recall = np.mean(metrics.box.mr)

f1_overall = 2 * precision * recall / (precision + recall + 1e-9)

print(f"mAP50     :   {map50:.4f}")
print(f"mAP50-95  :   {map5095:.4f}")
print(f"Precision :   {precision:.4f}")
print(f"Recall    :   {recall:.4f}")
print(f"F1-score  :   {f1_overall:.4f}")

print("\n\n===== PER-CLASS METRICS =====")

names = metrics.names  # class names dict

for i, name in names.items():
    p = metrics.box.p[i]
    r = metrics.box.r[i]
    f1 = 2 * p * r / (p + r + 1e-9)

    print(f"\nClass: {name}")
    print(f"  mAP50     :   {metrics.box.ap50[i]:.4f}")
    print(f"  mAP50-95  :   {metrics.box.ap[i]:.4f}")
    print(f"  Precision :   {p:.4f}")
    print(f"  Recall    :   {r:.4f}")
    print(f"  F1-score  :   {f1:.4f}")



===== OVERALL METRICS (2 Stage) =====
mAP50     :   0.6696
mAP50-95  :   0.4403
Precision :   0.7278
Recall    :   0.6292
F1-score  :   0.6750


===== PER-CLASS METRICS =====

Class: motorcycle
  mAP50     :   0.5137
  mAP50-95  :   0.2309
  Precision :   0.6158
  Recall    :   0.5147
  F1-score  :   0.5607

Class: car
  mAP50     :   0.8988
  mAP50-95  :   0.6127
  Precision :   0.8916
  Recall    :   0.8440
  F1-score  :   0.8672

Class: bus
  mAP50     :   0.6829
  mAP50-95  :   0.5177
  Precision :   0.7892
  Recall    :   0.5835
  F1-score  :   0.6709

Class: truck
  mAP50     :   0.5830
  mAP50-95  :   0.3998
  Precision :   0.6148
  Recall    :   0.5748
  F1-score  :   0.5941


In [ ]:
#Print rapi
import numpy as np
m1 = YOLO(str(best1))

metrics1 = m1.val(
    data=str(DATA_YAML),   # evaluasi balik ke dataset utama
    split="test",
    imgsz=1280,
    conf=0.001,
    iou=0.6,
    verbose=True
)

print("\n===== OVERALL METRICS (1 Stage) =====")

map50 = np.mean(metrics1.box.map50)
map5095 = np.mean(metrics1.box.map)
precision = np.mean(metrics1.box.mp)
recall = np.mean(metrics1.box.mr)

f1_overall = 2 * precision * recall / (precision + recall + 1e-9)

print(f"mAP50     :   {map50:.4f}")
print(f"mAP50-95  :   {map5095:.4f}")
print(f"Precision :   {precision:.4f}")
print(f"Recall    :   {recall:.4f}")
print(f"F1-score  :   {f1_overall:.4f}")

print("\n\n===== PER-CLASS METRICS =====")

names = metrics1.names  # class names dict

for i, name in names.items():
    p = metrics1.box.p[i]
    r = metrics1.box.r[i]
    f1 = 2 * p * r / (p + r + 1e-9)

    print(f"\nClass: {name}")
    print(f"  mAP50     :   {metrics1.box.ap50[i]:.4f}")
    print(f"  mAP50-95  :   {metrics1.box.ap[i]:.4f}")
    print(f"  Precision :   {p:.4f}")
    print(f"  Recall    :   {r:.4f}")
    print(f"  F1-score  :   {f1:.4f}")


Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11m summary (fused): 126 layers, 20,033,116 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 105.9±48.2 MB/s, size: 205.0 KB)
val: Scanning /content/drive/MyDrive/VMERGED_RES/labels/test.cache... 1591 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1591/1591 417.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 100/100 4.6it/s 21.7s
                   all       1591      45513      0.732      0.627       0.67      0.439
            motorcycle        802       5883      0.617      0.504      0.511      0.229
                   car       1546      34023      0.882      0.853      0.899       0.61
                   bus        843       2945      0.798      0.585      0.685      0.517
                 truck        753       2662       0.63      0.568      0.586      0.399
Speed: 1.

In [ ]:
m.export(
    format="engine",
    half=True,
    project="/content/drive/MyDrive/runs_yolo2",
    name="y11m_2stg"
)


WARNING ⚠️ TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)

PyTorch: starting from '/content/drive/MyDrive/runs_yolo2/stage2_y11m_1280_focusMT/weights/best.pt' with input shape (1, 3, 1280, 1280) BCHW and output shape(s) (1, 8, 33600) (38.8 MB)

ONNX: starting export with onnx 1.20.1 opset 20...
ONNX: slimming with onnxslim 0.1.85...
ONNX: export success ✅ 3.0s, saved as '/content/drive/MyDrive/runs_yolo2/stage2_y11m_1280_focusMT/weights/best.onnx' (77.2 MB)
requirements: Ultralytics requirement ['tensorrt-cu12>=7.0.0,!=10.1.0'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 3 packages in 33.79s
Prepared 3 packages in 20.95s
Installed 3 packages in 1ms
 + tensorrt-cu12==10.15.1.29
 + tensorrt-cu12-bindings==10.15.1.29
 + tensorrt-cu12-libs==10.15.1.29

requirements: AutoUpdate success ✅ 55.4s
WARNING ⚠️ requirements: Restart runtime or re

'/content/drive/MyDrive/runs_yolo2/stage2_y11m_1280_focusMT/weights/best.engine'